In [ ]:
%config InlineBackend.figure_formats = ['svg'] 
# %%
import glob
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-darkgrid")

# %%
# Helper: load all monomer_i.csv files and the average.csv

# Find monomer CSVs
monomer_files = sorted(glob.glob("monomer_*.csv"))

if not monomer_files:
    raise FileNotFoundError("No 'monomer_*.csv' files found in the current directory.")

# Extract monomer index from filename using regex
monomer_pattern = re.compile(r"monomer_(\d+)\.csv")

monomer_dfs = []
for f in monomer_files:
    m = monomer_pattern.search(os.path.basename(f))
    if not m:
        continue
    monomer_id = int(m.group(1))
    df = pd.read_csv(f)
    # Expect columns: t, twist_deg, bend_deg
    if not {"t", "twist_deg", "bend_deg"}.issubset(df.columns):
        raise ValueError(f"File {f} does not have expected columns t, twist_deg, bend_deg.")
    df["monomer"] = monomer_id
    monomer_dfs.append(df)

if not monomer_dfs:
    raise RuntimeError("No valid monomer CSVs loaded.")

monomers = pd.concat(monomer_dfs, ignore_index=True).sort_values(["monomer", "t"])

# Load average.csv
avg_file = "average.csv"
if not os.path.exists(avg_file):
    raise FileNotFoundError("average.csv not found in the current directory.")

avg = pd.read_csv(avg_file)
# Expect columns: t, twist_avg_deg, bend_avg_deg
if not {"t", "twist_avg_deg", "bend_avg_deg"}.issubset(avg.columns):
    raise ValueError("average.csv does not have expected columns t, twist_avg_deg, bend_avg_deg.")

# %%
# Plot 1: Twist – all monomers in one figure

fig, ax = plt.subplots(figsize=(8, 5))

for monomer_id, df_m in monomers.groupby("monomer"):
    ax.plot(df_m["t"], df_m["twist_deg"], label=f"Monomer {monomer_id}", alpha=0.2)

ax.set_xlabel("Timestep")
ax.set_ylabel("Twist (deg)")
ax.set_title("Twist per monomer")
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

# %%
# Plot 2: Bend – all monomers in one figure

fig, ax = plt.subplots(figsize=(8, 5))

for monomer_id, df_m in monomers.groupby("monomer"):
    ax.plot(df_m["t"], df_m["bend_deg"], label=f"Monomer {monomer_id}", alpha=0.2)

ax.set_xlabel("Timestep")
ax.set_ylabel("Bend (deg)")
ax.set_title("Bend per monomer")
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

# %%
# Plot 3: Average twist

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(avg["t"], avg["twist_avg_deg"], color="blue",alpha = 0.5, label="Average twist")

ax.set_xlabel("Timestep")
ax.set_ylabel("Twist (deg)")
ax.set_title("Average twist across monomers")
ax.legend()
plt.tight_layout()
plt.show()

# %%
# Plot 4: Average bend

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(avg["t"], avg["bend_avg_deg"], color="blue", alpha = 0.5, label="Average bend")

ax.set_xlabel("Timestep")
ax.set_ylabel("Bend (deg)")
ax.set_title("Average bend across monomers")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns  # for KDE plots

# Use a colormap for monomer curves
cmap = plt.cm.get_cmap("tab10")

# %%
# Plot 5: Twist distribution per monomer (KDE with fill and mean lines)

fig, ax = plt.subplots(figsize=(8, 5))

for i, (monomer_id, df_m) in enumerate(monomers.groupby("monomer")):
    values = df_m["twist_deg"].dropna().values
    if len(values) < 2:
        continue  # KDE needs at least 2 points

    mean_val = np.nanmean(values)
    color = cmap(i % cmap.N)

    # KDE curve with filled area
    sns.kdeplot(
        x=values,
        ax=ax,
        fill=False,
        alpha=0.8,
        linewidth=1.5,
        color=color,
        label=f"Monomer {monomer_id} (mean={mean_val:.2f}°)",
        bw_adjust=0.5,
    )



ax.set_xlabel("Twist (deg)")
ax.set_ylabel("Probability density")
ax.set_title("Twist distribution per monomer across trajectory")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# %%
# Plot 6: Bend distribution per monomer (KDE with fill and mean lines)

fig, ax = plt.subplots(figsize=(8, 5))

for i, (monomer_id, df_m) in enumerate(monomers.groupby("monomer")):
    values = df_m["bend_deg"].dropna().values
    if len(values) < 2:
        continue

    mean_val = np.nanmean(values)
    color = cmap(i % cmap.N)

    sns.kdeplot(
        x=values,
        ax=ax,
        fill=False,
        alpha=0.8,
        linewidth=1.5,
        color=color,
        label=f"Monomer {monomer_id} (mean={mean_val:.2f}°)",
        bw_adjust=0.5,
    )



ax.set_xlabel("Bend (deg)")
ax.set_ylabel("Probability density")
ax.set_title("Bend distribution per monomer across trajectory")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("mono_bent_K1IVT8mer.svg", format="svg")
plt.show()


# %%
# Plot 7: Distribution of time-averaged twist (KDE + mean and mean±std)

fig, ax = plt.subplots(figsize=(8, 4))

twist_avg_vals = avg["twist_avg_deg"].dropna().values
mean_twist_avg = np.nanmean(twist_avg_vals)
std_twist_avg = np.nanstd(twist_avg_vals)

sns.kdeplot(
    x=twist_avg_vals,
    ax=ax,
    fill=True,
    alpha=0.2,
    color="tab:blue",
    linewidth=1.5,
    label=f"mean = {mean_twist_avg:.2f}°, std = {std_twist_avg:.2f}°",
    bw_adjust=0.3,
)

# Mean line
ax.axvline(
    mean_twist_avg,
    color="k",
    linestyle="-",
    linewidth=1.5,
    label="mean",
)

# Mean ± std lines
ax.axvline(
    mean_twist_avg + std_twist_avg,
    color="k",
    linestyle="--",
    linewidth=1.0,
    label="mean ± std",
)
ax.axvline(
    mean_twist_avg - std_twist_avg,
    color="k",
    linestyle="--",
    linewidth=1.0,
)

ax.set_xlabel("Average twist (deg)")
ax.set_ylabel("Probability density")
ax.set_title("Distribution of average twist across trajectory")
ax.legend()
plt.tight_layout()
plt.show()

# %%
# Plot 8: Distribution of time-averaged bend (KDE + mean and mean±std)

fig, ax = plt.subplots(figsize=(8, 4))

bend_avg_vals = avg["bend_avg_deg"].dropna().values
mean_bend_avg = np.nanmean(bend_avg_vals)
std_bend_avg = np.nanstd(bend_avg_vals)

sns.kdeplot(
    x=bend_avg_vals,
    ax=ax,
    fill=True,
    alpha=0.2,
    color="tab:blue",
    linewidth=1.5,
    label=f"mean = {mean_bend_avg:.2f}°, std = {std_bend_avg:.2f}°",
    bw_adjust=0.3,
)

# Mean line
ax.axvline(
    mean_bend_avg,
    color="k",
    linestyle="-",
    linewidth=1.5,
    label="mean",
)

# Mean ± std lines
ax.axvline(
    mean_bend_avg + std_bend_avg,
    color="k",
    linestyle="--",
    linewidth=1.0,
    label="mean ± std",
)
ax.axvline(
    mean_bend_avg - std_bend_avg,
    color="k",
    linestyle="--",
    linewidth=1.0,
)

ax.set_xlabel("Average bend (deg)")
ax.set_ylabel("Probability density")
ax.set_title("Distribution of average bend across trajectory")
ax.legend()
plt.tight_layout()
plt.savefig("multi_bent_K1IVT8mer.svg", format="svg")
plt.show()
